In [1]:
!pip install faiss-cpu rank_bm25 sentence-transformers datasets transformers \
             rouge-score bert-score nltk sacrebleu scipy --quiet
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("All deps ready")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 46.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.6 MB/s eta 0:00:00
All deps ready


In [2]:
from huggingface_hub import login
import os
from getpass import getpass

hf_token = os.environ.get("HF_TOKEN", "")
if not hf_token:
    hf_token = getpass("HuggingFace token: ")
login(hf_token)


HuggingFace token:  ········


In [3]:
from datasets import load_dataset
print("Loading TechQA KB...")
techqa_raw = load_dataset("nvidia/TechQA-RAG-Eval", split="train")
tech_data  = []   # KB chunks for retrieval
eval_qa    = []   # unique (question, answer, relevant_ids) for evaluation
seen_text  = set()
seen_qa    = set()
text_to_id = {}   # text_key -> doc_id

# First pass: register all unique chunks
for item in techqa_raw:
    q = item["question"].strip()
    for ctx in item["contexts"]:
        txt = ctx["text"].strip()
        k   = txt[:300]
        if k in seen_text or len(txt) < 30:
            continue
        seen_text.add(k)
        doc_id = f"tech_{len(tech_data)}"
        text_to_id[k] = doc_id
        tech_data.append({
            "text"  : txt,
            "title" : ctx.get("title", q[:80]),
            "source": "TechQA",
            "domain": "it_tech",
            "doc_id": doc_id,
        })

# Second pass: build eval_qa with relevant_ids
for item in techqa_raw:
    q = item["question"].strip()
    a = (item.get("answer") or "").strip()
    if not (q and a) or q in seen_qa:
        continue
    seen_qa.add(q)
    relevant_ids = set()
    for ctx in item["contexts"]:
        k = ctx["text"].strip()[:300]
        if k in text_to_id:
            relevant_ids.add(text_to_id[k])
    eval_qa.append({
        "question"    : q,
        "answer"      : a,
        "relevant_ids": relevant_ids,
    })

print(f"TechQA KB : {len(tech_data)} unique chunks")
print(f"Eval pairs: {len(eval_qa)} unique question-answer pairs")
print(f"Avg relevant docs per question: "
      f"{sum(len(x['relevant_ids']) for x in eval_qa)/len(eval_qa):.1f}")


Loading TechQA KB...


README.md: 0.00B [00:00, ?B/s]

train.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/910 [00:00<?, ? examples/s]

TechQA KB : 492 unique chunks
Eval pairs: 890 unique question-answer pairs
Avg relevant docs per question: 0.7


In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np, faiss
from rank_bm25 import BM25Okapi

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def tokenize(text):
    return text.lower().split()

def build_kb_index(data, name):
    texts = [d["text"] for d in data if d.get("text")]
    print(f"  Embedding {name} ({len(texts)} chunks)...")
    emb = model.encode(texts, batch_size=64, show_progress_bar=False, normalize_embeddings=True)
    emb = np.array(emb).astype("float32")
    idx = faiss.IndexFlatIP(emb.shape[1])
    idx.add(emb)
    bm25 = BM25Okapi([tokenize(t) for t in texts])
    return {"index": idx, "bm25": bm25, "texts": texts, "data": data, "name": name}

print("Building TechQA index...")
KB = {"it_tech": build_kb_index(tech_data, "IT/Tech (TechQA)")}
print("TechQA KB indexed")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building TechQA index...
  Embedding IT/Tech (TechQA) (492 chunks)...
TechQA KB indexed


In [5]:
from sentence_transformers import CrossEncoder

def hybrid_search(query, kb, top_k=15):
    index, _bm25, _meta = kb["index"], kb["bm25"], kb["data"]
    q_emb = model.encode([query], normalize_embeddings=True).astype("float32")

    _, indices = index.search(q_emb, top_k)
    dense_rank = {int(indices[0][r]): r for r in range(len(indices[0]))}

    bm25_scores = _bm25.get_scores(query.lower().split())
    bm25_top    = np.argsort(bm25_scores)[::-1][:top_k]
    bm25_rank   = {int(idx): r for r, idx in enumerate(bm25_top)}

    k = 60
    all_ids = set(dense_rank) | set(bm25_rank)
    rrf = {i: (1/(k+dense_rank[i]) if i in dense_rank else 0)
             + (1/(k+bm25_rank[i])  if i in bm25_rank  else 0)
           for i in all_ids}

    top_ids = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
    return [_meta[i] for i in top_ids]


reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, docs, top_k=5):
    if not docs:
        return []
    scores = reranker.predict([[query, d["text"]] for d in docs])
    return [d for d, _ in sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)][:top_k]


def retrieve(query, top_k_per_kb=15, rerank_top_k=7):
    all_docs = []
    for key in ["it_tech"]:
        docs = hybrid_search(query, KB[key], top_k=top_k_per_kb)
        for d in docs:
            d["domain"] = key
        all_docs.extend(docs)

    seen, unique = set(), []
    for d in all_docs:
        key = d["text"][:200]
        if key not in seen:
            seen.add(key)
            unique.append(d)

    return rerank(query, unique, top_k=rerank_top_k)

print("Hybrid + RRF + Reranker pipeline ready")


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Hybrid + RRF + Reranker pipeline ready


In [6]:
def build_context_with_citations(docs, max_chars=500):
    context_lines, citations = [], []
    
    for i, doc in enumerate(docs, 1):
        ref = f"[{i}]"
        snippet = doc.get("text", "")[:max_chars].strip()
        
        context_lines.append(
            f"{ref} ({doc.get('domain','?')}) {doc.get('title','')}\n{snippet}"
        )
        
        citations.append({
            "ref": ref,
            "doc_id": doc.get("doc_id", ""),
            "title": doc.get("title", "Unknown"),
            "source": doc.get("source", "Unknown"),
            "domain": doc.get("domain", "unknown"),
            "snippet": snippet,
            "rank": i
        })
    
    return "\n\n".join(context_lines), citations


print("Citation builder ready")

Citation builder ready


In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch, re

_gen_model_name = "google/flan-t5-large"
_gen_tokenizer = _gen_model = _gen_device = None

def _load_generator():
    global _gen_tokenizer, _gen_model, _gen_device
    if _gen_model is None:
        print(f"Loading {_gen_model_name}...")
        _gen_tokenizer = AutoTokenizer.from_pretrained(_gen_model_name)
        _gen_model     = AutoModelForSeq2SeqLM.from_pretrained(_gen_model_name)
        _gen_device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        _gen_model     = _gen_model.to(_gen_device)
        print(f"Generator loaded on {_gen_device}")

# ── Strategy: Sentence-index extraction ──────────────────────────────────────
# 1. Split all retrieved doc text into numbered sentences
# 2. Ask flan-t5 to output ONLY the sentence numbers that answer the question
# 3. Return the full original sentences (guarantees correct length + faithfulness)

import re as _re

def _split_sentences(text):
    """Split text into sentences, return list of non-empty strings."""
    raw = _re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in raw if len(s.strip()) > 15]

def _build_numbered_context(docs, max_chars=600):
    """
    Returns:
        numbered_text : context with [S1] [S2] ... sentence labels
        sent_map      : {index: full_sentence_text}
    """
    sent_map   = {}
    lines      = []
    idx        = 1
    for doc in docs:
        for sent in _split_sentences(doc.get("text","")[:max_chars]):
            sent_map[idx] = sent
            lines.append(f"[S{idx}] {sent}")
            idx += 1
    return "\n".join(lines), sent_map

# Prompt: ask model to output only sentence IDs
EXTRACT_PROMPT = (
    "Below are numbered sentences from technical documents.\n"
    "List the sentence numbers (e.g. S1, S3) that directly answer the question.\n"
    "Output ONLY the sentence numbers, separated by commas. Nothing else.\n\n"
    "Sentences:\n{context}\n\n"
    "Question: {question}\n\n"
    "Relevant sentence numbers:"
)

def _generate_text(prompt, max_new_tokens=60):
    _load_generator()
    inputs = _gen_tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=2048,
    ).to(_gen_device)
    with torch.no_grad():
        outputs = _gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=False,   # FIXED: was True, caused truncation
            no_repeat_ngram_size=2,
            length_penalty=0.8,     # slight brevity for ID lists
        )
    return _gen_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

def _parse_ids(raw_output, sent_map):
    """Extract sentence IDs from model output like 'S1, S3, S5'."""
    ids = [int(x) for x in _re.findall(r'S(\d+)', raw_output)
           if int(x) in sent_map]
    return ids

def generate(query, docs):
    if not docs:
        return "No relevant documents found.", []

    numbered_ctx, sent_map = _build_numbered_context(docs, max_chars=600)
    _, citations = build_context_with_citations(docs, max_chars=600)

    prompt = EXTRACT_PROMPT.format(context=numbered_ctx, question=query)
    raw    = _generate_text(prompt)

    ids = _parse_ids(raw, sent_map)

    if ids:
        # Return the full extracted sentences joined together
        answer = " ".join(sent_map[i] for i in sorted(ids))
    else:
        # Fallback: return top-1 doc's first two sentences
        fallback_sents = _split_sentences(docs[0].get("text",""))[:2]
        answer = " ".join(fallback_sents) if fallback_sents else "No answer found."

    used = [c for c in citations if c["ref"] in answer]
    return answer, used

print(f"Generator ready — sentence-index extraction ({_gen_model_name})")


Generator ready — sentence-index extraction (google/flan-t5-large)


In [8]:
from sentence_transformers import CrossEncoder
import re, numpy as np
from scipy.special import softmax

print("Loading NLI model...")
nli_model = CrossEncoder("cross-encoder/nli-deberta-v3-small", num_labels=3)
print("NLI model ready")

def check_faithfulness(answer, docs,
                       entail_threshold=0.40,
                       contradiction_threshold=0.40):
    """
    Sentence-level NLI faithfulness.
    nli-deberta-v3-small label order: entailment=0, neutral=1, contradiction=2
    Scores softmaxed from raw logits.
    """
    if not docs or not answer.strip():
        return {"overall": 0.0, "sentences": [], "flag": True}

    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', answer)
                 if len(s.strip()) > 20]
    if not sentences:
        return {"overall": 1.0, "sentences": [], "flag": False}

    results = []
    for sent in sentences:
        pairs  = [[d["text"][:500], sent] for d in docs]
        logits = nli_model.predict(pairs)
        probs  = softmax(logits, axis=1)
        best   = int(np.argmax(probs[:, 0]))
        ent    = float(probs[best, 0])
        con    = float(probs[best, 2])
        verdict = ("contradiction" if con > contradiction_threshold
                   else "grounded"   if ent >= entail_threshold
                   else "weak_support")
        results.append({"sentence": sent, "entailment": ent,
                         "contradiction": con, "verdict": verdict})

    overall = float(np.mean([r["entailment"] for r in results]))
    flagged = (any(r["verdict"] == "contradiction" for r in results)
               or overall < entail_threshold)
    return {"overall": overall, "sentences": results, "flag": flagged}


Loading NLI model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/568M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

NLI model ready


In [9]:
def run_full_pipeline(query):
    if not isinstance(query, str) or not query.strip():
        raise ValueError("Query must be a non-empty string")
    print("\n=== QUERY ==="); print(query)

    docs = retrieve(query)
    if not docs:
        return {"answer": "No relevant documents found.", "citations": [], "faithfulness": None}

    answer, citations = generate(query, docs)
    faith = check_faithfulness(answer, docs)

    print("\n=== ANSWER ==="); print(answer)
    print("\n=== FAITHFULNESS SCORE ==="); print(round(faith["overall"], 3))
    return {"answer": answer, "citations": citations, "faithfulness": faith}

print("Full pipeline ready")


Full pipeline ready


In [10]:
run_full_pipeline("How do I fix a job submission error in IBM Streams?")



=== QUERY ===
How do I fix a job submission error in IBM Streams?
Loading google/flan-t5-large...


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generator loaded on cpu

=== ANSWER ===
DIAGNOSING THE PROB

=== FAITHFULNESS SCORE ===
1.0


{'answer': 'DIAGNOSING THE PROB',
 'citations': [],
 'faithfulness': {'overall': 1.0, 'sentences': [], 'flag': False}}

In [11]:
# ============================================================
# FULL RAG EVALUATION
# Metrics: EM, Partial-EM, F1, Accuracy, Token Precision/Recall
#          ROUGE-1/2/L, BLEU, BERTScore F1,
#          Faithfulness (NLI), Hallucination Rate,
#          MRR, Precision@k, Recall@k, Hit@k, NDCG@k
# ============================================================

assert eval_qa, "eval_qa is empty -- re-run cell 2 first"
assert "relevant_ids" in eval_qa[0], "eval_qa missing relevant_ids -- re-run cell 2"

from rouge_score import rouge_scorer as rouge_module
from collections import defaultdict
import numpy as np, string
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score_fn


# ── Text normalisation (SQuAD-style) ─────────────────────────────────────────
def _norm(text):
    text = text.lower().translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())

def exact_match(pred, gold):
    """Strict EM: full normalised string must match."""
    return int(_norm(pred) == _norm(gold))

def partial_em(pred, gold, n_words=10):
    """
    Relaxed EM: check if any n-word window of gold appears in pred.
    Helps when gold is a long paragraph but model copies a key sentence.
    """
    g_toks = _norm(gold).split()
    p_norm = _norm(pred)
    for i in range(max(1, len(g_toks) - n_words + 1)):
        window = " ".join(g_toks[i:i + n_words])
        if window in p_norm:
            return 1
    return 0

def token_f1(pred, gold):
    """Token-overlap F1 (SQuAD style)."""
    p_toks = _norm(pred).split()
    g_toks = _norm(gold).split()
    common = set(p_toks) & set(g_toks)
    if not common:
        return 0.0
    prec = len(common) / len(p_toks) if p_toks else 0.0
    rec  = len(common) / len(g_toks) if g_toks else 0.0
    return (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0

def token_precision(pred, gold):
    p_toks = _norm(pred).split()
    g_set  = set(_norm(gold).split())
    return (sum(1 for t in p_toks if t in g_set) / len(p_toks)) if p_toks else 0.0

def token_recall(pred, gold):
    g_toks = _norm(gold).split()
    p_set  = set(_norm(pred).split())
    return (sum(1 for t in g_toks if t in p_set) / len(g_toks)) if g_toks else 0.0

def compute_ndcg(retrieved_ids, relevant_ids, k):
    dcg  = sum(1.0/np.log2(r+1) for r,did in enumerate(retrieved_ids[:k],1) if did in relevant_ids)
    idcg = sum(1.0/np.log2(r+1) for r in range(1, min(len(relevant_ids),k)+1))
    return dcg/idcg if idcg > 0 else 0.0

HALL_THRESHOLD = 0.40   # must match entail_threshold in check_faithfulness

def hallucination_flag(answer, docs):
    """1 = hallucinated (faithfulness < HALL_THRESHOLD)."""
    return int(check_faithfulness(answer, docs)["overall"] < HALL_THRESHOLD)


# ── Scorers ───────────────────────────────────────────────────────────────────
# IMPORTANT: rouge_scorer.score(target, prediction) -- target first!
rscorer     = rouge_module.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=True)
bleu_metric = BLEU(effective_order=True)

# ── Accumulators ─────────────────────────────────────────────────────────────
K_VALUES = [1, 3, 5, 7]
recall_at    = defaultdict(list)
precision_at = defaultdict(list)
ndcg_at      = defaultdict(list)
hit_at       = defaultdict(list)
mrr_scores   = []

em_list=[]; partial_em_list=[]; f1_list=[]
acc_list=[]; prec_list=[]; rec_list=[]
r1_list=[]; r2_list=[]; rL_list=[]; bleu_list=[]
faith_list=[]; hall_list=[]
pred_lens=[]; gold_lens=[]
all_preds=[]; all_refs=[]
skipped = 0

eval_items = eval_qa[:50]
print(f"Evaluating on {len(eval_items)} samples...\n")

for idx, item in enumerate(eval_items):
    query        = item["question"]
    ref_answer   = item["answer"]
    relevant_ids = item["relevant_ids"]

    if not relevant_ids:
        skipped += 1
        continue

    docs = retrieve(query, rerank_top_k=7)
    if not docs:
        skipped += 1
        continue

    retrieved_ids = [d["doc_id"] for d in docs]

    # ── MRR ──────────────────────────────────────────────────────────────────
    mrr_scores.append(next(
        (1.0/r for r,did in enumerate(retrieved_ids,1) if did in relevant_ids), 0.0))

    # ── Retrieval @ k ─────────────────────────────────────────────────────────
    for k in K_VALUES:
        hits = len(set(retrieved_ids[:k]) & relevant_ids)
        precision_at[k].append(hits / k)
        recall_at[k].append(hits / len(relevant_ids))
        hit_at[k].append(1 if hits > 0 else 0)
        ndcg_at[k].append(compute_ndcg(retrieved_ids, relevant_ids, k))

    # ── Generate ──────────────────────────────────────────────────────────────
    answer, _ = generate(query, docs[:5])

    # ── Length stats ──────────────────────────────────────────────────────────
    pred_lens.append(len(answer.split()))
    gold_lens.append(len(ref_answer.split()))

    # ── EM / Partial-EM / F1 / Accuracy / Precision / Recall ──────────────────
    em = exact_match(answer, ref_answer)
    em_list.append(em)
    acc_list.append(em)
    partial_em_list.append(partial_em(answer, ref_answer))
    f1_list.append(token_f1(answer, ref_answer))
    prec_list.append(token_precision(answer, ref_answer))
    rec_list.append(token_recall(answer, ref_answer))

    # ── ROUGE (score(target, prediction)) ─────────────────────────────────────
    rs = rscorer.score(ref_answer, answer)
    r1_list.append(rs["rouge1"].fmeasure)
    r2_list.append(rs["rouge2"].fmeasure)
    rL_list.append(rs["rougeL"].fmeasure)

    # ── BLEU ──────────────────────────────────────────────────────────────────
    try:
        bleu_list.append(bleu_metric.sentence_score(answer, [ref_answer]).score / 100.0)
    except Exception:
        bleu_list.append(0.0)

    # ── Faithfulness & Hallucination (reuse single call) ─────────────────────
    faith = check_faithfulness(answer, docs[:5])
    faith_list.append(faith["overall"])
    hall_list.append(int(faith["overall"] < HALL_THRESHOLD))

    all_preds.append(answer)
    all_refs.append(ref_answer)

    if (idx+1) % 10 == 0:
        print(f"  [{idx+1}/{len(eval_items)}] running... "
              f"(avg F1 so far: {np.mean(f1_list):.3f})")


# ── BERTScore (batched) ───────────────────────────────────────────────────────
print("\nComputing BERTScore (distilbert, ~30-60s on CPU)...")
try:
    _, _, F_bs = bert_score_fn(
        all_preds, all_refs,
        lang="en",
        model_type="distilbert-base-uncased",
        verbose=False,
        batch_size=16,
    )
    bs_list = F_bs.numpy().tolist()
except Exception as e:
    print(f"  BERTScore failed: {e}")
    bs_list = [0.0] * len(all_preds)


# ── Report ────────────────────────────────────────────────────────────────────
n  = len(mrr_scores)
ng = len(f1_list)
W  = 65
SEP  = "=" * W
SEP2 = "-" * W

print(f"\n{SEP}")
print(f"  TechQA RAG Evaluation  |  {n} evaluated  |  {skipped} skipped")
print(SEP)

print("\n  RETRIEVAL METRICS")
print(SEP2)
print(f"  {'Metric':<30}  {'Value':>8}")
print(SEP2)
print(f"  {'MRR':<30}  {np.mean(mrr_scores):>8.4f}")
for k in K_VALUES:
    if k != K_VALUES[0]:
        print()
    print(f"  {'Precision@'+str(k):<30}  {np.mean(precision_at[k]):>8.4f}")
    print(f"  {'Recall@'+str(k):<30}  {np.mean(recall_at[k]):>8.4f}")
    print(f"  {'Hit@'+str(k):<30}  {np.mean(hit_at[k]):>8.4f}")
    print(f"  {'NDCG@'+str(k):<30}  {np.mean(ndcg_at[k]):>8.4f}")

print(f"\n  GENERATION METRICS  (n={ng})")
print(SEP2)
print(f"  {'Metric':<30}  {'Value':>8}  Notes")
print(SEP2)
print(f"  {'Avg pred length (words)':<30}  {np.mean(pred_lens):>8.1f}")
print(f"  {'Avg gold length (words)':<30}  {np.mean(gold_lens):>8.1f}")
print(SEP2)
print(f"  {'Exact Match (EM)':<30}  {np.mean(em_list):>8.4f}  strict")
print(f"  {'Partial EM (10-gram)':<30}  {np.mean(partial_em_list):>8.4f}  relaxed")
print(f"  {'Accuracy (= strict EM)':<30}  {np.mean(acc_list):>8.4f}")
print(f"  {'Token F1':<30}  {np.mean(f1_list):>8.4f}")
print(f"  {'Token Precision':<30}  {np.mean(prec_list):>8.4f}")
print(f"  {'Token Recall':<30}  {np.mean(rec_list):>8.4f}")
print(SEP2)
print(f"  {'ROUGE-1':<30}  {np.mean(r1_list):>8.4f}")
print(f"  {'ROUGE-2':<30}  {np.mean(r2_list):>8.4f}")
print(f"  {'ROUGE-L':<30}  {np.mean(rL_list):>8.4f}")
print(f"  {'BLEU':<30}  {np.mean(bleu_list):>8.4f}")
print(f"  {'BERTScore F1':<30}  {np.mean(bs_list):>8.4f}  semantic similarity")
print(SEP2)
print(f"  {'Faithfulness (NLI)':<30}  {np.mean(faith_list):>8.4f}  higher=better")
hn = sum(hall_list); hd = len(hall_list)
print(f"  {'Hallucination Rate':<30}  {np.mean(hall_list):>8.4f}  ({hn}/{hd} flagged, lower=better)")
print(SEP)


Evaluating on 50 samples...

  [10/50] running... (avg F1 so far: 0.116)
  [20/50] running... (avg F1 so far: 0.123)
  [30/50] running... (avg F1 so far: 0.130)
  [40/50] running... (avg F1 so far: 0.148)
  [50/50] running... (avg F1 so far: 0.158)

Computing BERTScore (distilbert, ~30-60s on CPU)...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  TechQA RAG Evaluation  |  41 evaluated  |  9 skipped

  RETRIEVAL METRICS
-----------------------------------------------------------------
  Metric                             Value
-----------------------------------------------------------------
  MRR                               0.7840
  Precision@1                       0.6829
  Recall@1                          0.6829
  Hit@1                             0.6829
  NDCG@1                            0.6829

  Precision@3                       0.2927
  Recall@3                          0.8780
  Hit@3                             0.8780
  NDCG@3                            0.8028

  Precision@5                       0.1756
  Recall@5                          0.8780
  Hit@5                             0.8780
  NDCG@5                            0.8028

  Precision@7                       0.1324
  Recall@7                          0.9268
  Hit@7                             0.9268
  NDCG@7                            0.8197

  GENERATION 